<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Triage de Urgencias: Decidir con Costos 🏥
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/Para%20Dummies/00_Casos_Estudio_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno integrador del Módulo 09 (Casos de estudio y aplicaciones)](../00_Casos_de_Estudio_y_Aplicaciones.ipynb). Un proyecto de *machine learning* se parece a lo que hace un **médico de urgencias**: no trata a todos por igual; **decide a quién atender primero**, sabiendo que **equivocarse en un sentido cuesta más que en el otro** (dejar ir a un enfermo grave es peor que revisar a uno sano). Lo veremos con una empresa que quiere **detectar a los clientes que se van a ir**.

Al terminar podrás explicar:
1. Por qué **«acertar mucho»** puede ser una trampa cuando lo que buscamos es **raro**.
2. Cómo el **dinero** (lo que cuesta cada error) decide **a partir de qué probabilidad actuar**.
3. Por qué siempre guardamos un **examen sellado** y por qué un modelo que **memoriza** engaña.

> 🔗 Si luego quieres ver el proyecto completo (con datos faltantes, comparación rigurosa de modelos, calibración, explicaciones y *ficha del modelo*): [cuaderno estándar](../00_Casos_de_Estudio_y_Aplicaciones.ipynb). Los otros cuadernos del módulo: [imágenes](01_Vision_Computador_Dummies.ipynb), [texto](02_NLP_Dummies.ipynb) y [recomendaciones](03_Recomendacion_Dummies.ipynb).

---
## 1. El guardia que dice «aquí no pasa nada» 😴

Una tienda tiene **800 clientes**. Cada uno tiene dos datos: **cuántos meses lleva sin comprar** y **cuántas quejas** ha presentado. Aproximadamente **1 de cada 10** se va. Queremos **adelantarnos** y llamar a los que están en riesgo para ofrecerles algo.

Primero, una **trampa**. Un guardia perezoso que repite *«nadie se va»* acierta **90 %** de las veces, ¡y no ha detectado a nadie! Cuando lo que buscamos es **raro**, el porcentaje de aciertos **no sirve** para saber si el modelo es bueno (en el [Módulo 03](../../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/00_Metricas_de_Clasificacion.ipynb) vimos la «paradoja de la exactitud»). Los datos de aquí son **inventados** (sintéticos).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(3)

# --- 800 clientes inventados: cuantos MESES llevan sin comprar y cuantas QUEJAS han presentado ---
n = 800
meses_sin_comprar = rng.gamma(2.0, 2.0, n)
quejas = rng.poisson(0.6, n)
riesgo = 1 / (1 + np.exp(-(-4.2 + 0.28 * meses_sin_comprar + 0.7 * quejas)))          # la probabilidad REAL de que se vaya
se_va = (rng.random(n) < riesgo).astype(int)
X = np.column_stack([meses_sin_comprar, quejas])

print(f"De {n} clientes, se van {se_va.sum()} ({se_va.mean():.0%}). Los demás se quedan.")

# Una prueba SELLADA: la guardamos en un cajón y no la miramos hasta el final
X_tr, X_te, y_tr, y_te, r_tr, r_te = train_test_split(X, se_va, riesgo, test_size=0.4, stratify=se_va, random_state=0)

# El "modelo perezoso": nunca llamar a nadie. ¿Cuántos aciertos tiene?
acierto_perezoso = (y_te == 0).mean()
modelo = LogisticRegression().fit(X_tr, y_tr)
acierto_modelo = modelo.score(X_te, y_te)
print(f"Modelo perezoso («nadie se va»): {acierto_perezoso:.0%} de aciertos")
print(f"Modelo de verdad (regresión logística): {acierto_modelo:.0%} de aciertos")
print("→ Los aciertos casi no cambian... pero el modelo perezoso NO detecta a ningún cliente que se va. Los aciertos engañan.")
assert acierto_perezoso > 0.85

---
**Lección:** el modelo de verdad y el guardia perezoso tienen **los mismos aciertos** (≈ 90 %), pero **no son lo mismo**: el segundo no sirve para nada. Más abajo veremos qué sí cuenta: **cuánto dinero** se gana o se pierde.

---

## 2. La cuenta que decide a quién llamar 💰

El modelo no dice «se va» o «se queda»: da una **probabilidad** (por ejemplo, «este cliente tiene 15 % de irse»). Entonces la pregunta es: **¿a partir de qué probabilidad hacemos la llamada?** La respuesta no es «50 %» por costumbre: la dan los **costos**.

- **Llamar** a alguien cuesta **50** (el tiempo y la oferta), se vaya o no.
- **Perder** a un cliente cuesta **500**.

Si un cliente tiene probabilidad $p$ de irse, **no llamarlo** cuesta en promedio $p\times 500$, y **llamarlo** cuesta $50$. Conviene llamar cuando $p\times 500>50$, o sea cuando **$p>0.10$**. Es como el triage de urgencias: **como perder un paciente grave es mucho peor que revisar a uno sano, se revisa a mucha gente con poca sospecha**. Comprobémoslo con el dinero real de la prueba.

In [ ]:
# El modelo da una PROBABILIDAD de que cada cliente se vaya. ¿A partir de que probabilidad hacemos la llamada?
p = modelo.predict_proba(X_te)[:, 1]
COSTO_LLAMADA, COSTO_PERDER = 50, 500            # llamar cuesta 50 (se vaya o no); perder a un cliente cuesta 500


def costo(umbral):
    llamar = p > umbral
    perdidos = ((y_te == 1) & ~llamar).sum()      # se fueron y NO los llamamos
    return perdidos * COSTO_PERDER + llamar.sum() * COSTO_LLAMADA


umbrales = np.linspace(0.01, 0.9, 90)
costos = np.array([costo(u) for u in umbrales])
mejor = umbrales[np.argmin(costos)]
costo_nadie = (y_te == 1).sum() * COSTO_PERDER
costo_todos = len(y_te) * COSTO_LLAMADA

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(umbrales, costos, color="#2563eb", lw=2.5, label="Costo según el umbral")
ax.axhline(costo_nadie, color="#dc2626", ls="--", label="No llamar a nadie")
ax.axhline(costo_todos, color="#f59e0b", ls="--", label="Llamar a todos")
ax.axvline(COSTO_LLAMADA / COSTO_PERDER, color="#16a34a", ls=":", lw=2, label="Umbral de la cuenta: 50/500 = 0.10")
ax.set_xlabel("Llamamos si la probabilidad de que se vaya es mayor que..."); ax.set_ylabel("Costo total"); ax.legend(fontsize=8)
ax.set_title("El mejor umbral lo decide el DINERO, no el 0.5", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

print(f"No llamar a nadie cuesta {costo_nadie:,.0f} | llamar a todos {costo_todos:,.0f} | llamar con umbral 0.50: {costo(0.5):,.0f} | con umbral 0.10: {costo(0.10):,.0f}")
print(f"El umbral con menor costo en esta prueba fue {mejor:.2f}, muy cerca de la cuenta 50/500 = 0.10.")
assert costo(0.10) < costo_nadie and costo(0.10) < costo(0.5)

---
**Lección:** llamar a **todos** o a **nadie** son las dos políticas «tontas» (cuestan 16 000 y 16 500). El modelo con el umbral que sale de la **cuenta** (0.10) las supera con claridad y también al umbral «de costumbre» de 0.50; el mínimo de la curva cae por ahí cerca (con una prueba tan pequeña, el punto exacto se mueve un poco; la cuenta 50/500 es más estable que lo que dice una sola muestra). **El negocio fija el umbral, no el algoritmo.**

---

## 3. El examen sellado y el aprendiz que memoriza 📝

Hay una regla de oro: antes de empezar, **se guarda en un cajón una parte de los datos** (el **examen sellado**) y no se mira hasta el final. Si no, es como dejar que el estudiante vea las preguntas del examen mientras estudia. Para verlo, comparemos a **dos aprendices**: uno que **memoriza** todo el material (un árbol sin límite) y uno que entiende la **idea general** (regresión logística).

In [ ]:
# Dos aprendices: uno MEMORIZA (arbol muy profundo) y otro entiende la idea general (regresion logistica). ¿Quien gana en la prueba sellada?
memorion = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)                  # sin limite de profundidad: se aprende todo de memoria
simple = LogisticRegression().fit(X_tr, y_tr)

filas = {"Memorión (árbol sin límite)": (memorion.score(X_tr, y_tr), memorion.score(X_te, y_te)),
         "Aprendiz sencillo (reg. logística)": (simple.score(X_tr, y_tr), simple.score(X_te, y_te))}
print(f"{'':<38}{'En lo que estudió':>20}{'En el examen sellado':>24}")
for nombre, (a, b) in filas.items():
    print(f"{nombre:<38}{a:>19.0%}{b:>24.0%}")

# Lo justo es comparar con las llamadas que SE ENCUENTRAN: de los clientes que se van, ¿a cuantos detecta cada uno con el umbral de costos?
for nombre, m in [("Memorión", memorion), ("Aprendiz sencillo", simple)]:
    llamar = m.predict_proba(X_te)[:, 1] > 0.10
    print(f"{nombre}: detecta {(llamar & (y_te == 1)).sum()} de {(y_te == 1).sum()} clientes que se van, llamando a {llamar.sum()} personas (costo {(((y_te == 1) & ~llamar).sum() * COSTO_PERDER + llamar.sum() * COSTO_LLAMADA):,.0f})")
assert filas["Memorión (árbol sin límite)"][0] > 0.99 and filas["Memorión (árbol sin límite)"][1] < filas["Memorión (árbol sin límite)"][0] - 0.05

---
**Lección:** el memorión saca **100 %** en lo que estudió y baja a ≈ 82 % en el examen sellado; además **casi no detecta** a los clientes que se van (5 de 33). El aprendiz sencillo saca ≈ 91 % y ≈ 90 %, **casi igual**, detecta bastantes más (17 de 33) y gasta menos. **Una nota alta en lo que se estudió no prueba nada; solo el examen sellado cuenta.** (Con una muestra tan pequeña las cifras cambian de una semilla a otra; lo que se mantiene es el patrón.)

---

## 4. La receta es la misma para cualquier proyecto 🍳

En el cuaderno estándar aplicamos **la misma receta** a tres proyectos distintos (**fuga de clientes**, **precio de una vivienda** y **grupos de clientes**). Los pasos son siempre estos:

| Paso | Analogía | Qué pasaría si te lo saltas |
|---|---|---|
| 1. **Definir la decisión y los costos** | El triage: ¿qué error cuesta más? | Optimizas lo que no importa |
| 2. **Guardar un examen sellado** | El examen que el estudiante no ve | Te engañas con una nota inflada |
| 3. **Probar primero lo más simple** | «¿Y si no hago nada?» | Presumes de un modelo que no mejora lo trivial |
| 4. **Comparar modelos con varios exámenes** | Varios parciales, no uno solo | La suerte decide al ganador |
| 5. **Elegir el umbral con el dinero** | La cuenta 50/500 | Llamas a demasiados o a muy pocos |
| 6. **Explicar y revisar los errores** | Preguntar *¿a quiénes les fallamos?* | Fallas siempre al mismo grupo sin enterarte |
| 7. **Documentar y vigilar** | La ficha del paciente | Nadie sabe para qué sirve el modelo ni cuándo falla |

Y un consejo de honestidad: **en los tres proyectos, el modelo simple ganó o empató** con el sofisticado. Lo moderno no siempre es lo mejor.

---
##### 🛠️ Práctica 1: Si llamar sale más caro, ¿cambia la cuenta?

Ahora la tienda tiene una oferta más cara: **llamar** a un cliente cuesta **100** (en vez de 50), y **perder** a un cliente sigue costando **500**. Usa las probabilidades `p` y las respuestas reales `y_te` que ya calculamos, y la constante `COSTO_PERDER`:

1. Haz la cuenta del triage: ¿a partir de qué probabilidad conviene llamar? Guarda el resultado en `umbral_nuevo` (costo de llamar ÷ costo de perder).
2. Escribe una función `costo_con(umbral, costo_llamada)` que devuelva el costo total de la prueba: los clientes que se van y **no** llamamos cuestan `COSTO_PERDER` cada uno, y cada llamada cuesta `costo_llamada`.
3. Calcula `costo_viejo` (con el umbral 0.10) y `costo_nuevo` (con `umbral_nuevo`), **los dos con la llamada a 100**, e imprímelos.
4. Comprueba que `umbral_nuevo` es 0.2 y que con el umbral nuevo no gastamos más que con el viejo.

In [ ]:
# Escribe tu código aquí

# 1) La cuenta del triage: llamar conviene cuando p * 500 > costo de llamar
# umbral_nuevo = ... / COSTO_PERDER

# 2) El costo total de la prueba
# def costo_con(umbral, costo_llamada):
#     llamar = p > umbral
#     perdidos = ((y_te == 1) & ~llamar).sum()
#     return perdidos * COSTO_PERDER + llamar.sum() * costo_llamada

# 3) Comparar el umbral viejo y el nuevo (con la llamada a 100)
# costo_viejo = costo_con(0.10, 100)
# costo_nuevo = costo_con(..., 100)
# print(costo_viejo, costo_nuevo)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) La cuenta del triage: llamar conviene cuando p * 500 > costo de llamar
umbral_nuevo = 100 / COSTO_PERDER

# 2) El costo total de la prueba
def costo_con(umbral, costo_llamada):
    llamar = p > umbral
    perdidos = ((y_te == 1) & ~llamar).sum()                 # se fueron y NO los llamamos
    return perdidos * COSTO_PERDER + llamar.sum() * costo_llamada

# 3) Comparar el umbral viejo y el nuevo (con la llamada a 100)
costo_viejo = costo_con(0.10, 100)
costo_nuevo = costo_con(umbral_nuevo, 100)
print(f"Umbral nuevo: {umbral_nuevo:.2f}")
print(f"Costo con el umbral viejo (0.10): {costo_viejo:,.0f} | con el umbral nuevo ({umbral_nuevo:.2f}): {costo_nuevo:,.0f}")

# 4) Comprobaciones
assert np.isclose(umbral_nuevo, 0.2)
assert costo_nuevo <= costo_viejo
print("\nSi llamar es más caro, solo vale la pena llamar a quienes tienen más riesgo: el umbral sube.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Un buen proyecto de *machine learning* no es «elegir el algoritmo más moderno»: es definir qué error cuesta más, guardar un examen sellado, comparar con lo más simple y dejar que el dinero decida a partir de qué probabilidad actuar.**

### Lo que aprendiste hoy:
- ✅ **Acertar mucho no basta** cuando lo buscado es raro: un guardia que dice «nadie se va» acierta 90 % y no sirve.
- ✅ **Los costos deciden el umbral:** si llamar cuesta 50 y perder 500, se llama cuando la probabilidad supera **0.10**, no 0.50.
- ✅ **El examen sellado** evita que te engañe el modelo que **memoriza** (100 % estudiando, ≈ 82 % en el examen).
- ✅ **La receta** (decisión, examen sellado, línea base, comparación, umbral, errores, ficha) sirve para **cualquier** proyecto.

> 🎓 **Siguiente paso:** [imágenes: la lupa que se desliza](01_Vision_Computador_Dummies.ipynb), o el [cuaderno estándar](../00_Casos_de_Estudio_y_Aplicaciones.ipynb) con el proyecto completo.

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué un modelo con 90 % de aciertos puede ser **inútil**? Piensa en otro caso de tu vida donde lo importante sea **raro** (un fraude, una enfermedad, un error de fábrica).
2. Si llamar costara 5 y perder un cliente costara 500, ¿el umbral sería más alto o más bajo que 0.10? ¿Y si llamar costara 400? Explica con la cuenta $p\times500>\text{costo de llamar}$.
3. ¿Qué es el «examen sellado» y qué le pasa a un estudiante (o a un modelo) que lo ve antes? ¿Qué señal te avisa de que un modelo está **memorizando**?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>